# **Fourth Homework Assignment** — Lenia with MPI

In [15]:
import numpy as np
import pandas as pd
import re

# Used for setting display settings to increase readability.
from IPython.core.display import display_html

In [16]:
# Custom formater.
def fromat_nan(val):
    if pd.isna(val):
        return ""
    if isinstance(val, float):
        return f"{round(val, 2)}"
    return f"{val}"

# Helper function that displays each dataframe in its own column.
def display_list(dfs, index=True, axis=-1, minmax="min"):
    # Convert each split to HTML with proper styling for side-by-side display.
    html_str = "<table><tr>" 
    for df in dfs:
        # Dataframe styling options.
        styled_df = df.style.format(fromat_nan)
        if axis >= 0 and minmax == "min":
            styled_df = styled_df.highlight_min(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_max(axis=axis, color='darkred')
        if axis >= 0 and minmax == "max":
            styled_df = styled_df.highlight_max(axis=axis, color='darkgreen')
            styled_df = styled_df.highlight_min(axis=axis, color='darkred')
        styled_html = styled_df.to_html(index=index)

        # Enables newlines and bold titles.
        name_html = df.attrs['name'].replace('\n', '<br>')
        name_html = f"<div style='text-align: left; font-weight: bold;'>{name_html}</div>"
        html_str += f"<td style='vertical-align: top; padding: 5px;'>{name_html + styled_html}</td>"
    html_str += "</tr></table>"
    display_html(html_str, raw=True)
    
# Helper function to display one data frame as multiple columns.
def display(df, cols, index=True):
    # Split the dataframe into columns.
    ratio = int(np.ceil(len(df) / cols))
    df_split = [df.iloc[i * ratio: (i + 1) * ratio] for i in range(cols)]
    display_list(df_split, index)


In [17]:
# Calculates means based on axis and returns the dataframe.
def get_means(dfs, axis):
    lst = []
    for df in dfs:
        mean = df.mean(axis).to_frame(name='Mean')
        mean.attrs['name'] = df.attrs['name']
        lst.append(mean)
    return lst

## **Time Comparisons**

In [18]:
import glob
from pathlib import Path

# CSV columns produced by benchmarks/bench.sh: Run,Size,Method,Procs,Nodes,Halo,Time
_csv = pd.concat([pd.read_csv(f) for f in glob.glob("results_*.csv")], ignore_index=True)
_csv.columns = [c.strip() for c in _csv.columns]
_csv["Time"] = pd.to_numeric(_csv["Time"], errors="coerce")
for c in ("Size", "Procs", "Nodes", "Halo"):
    _csv[c] = pd.to_numeric(_csv[c], errors="coerce").astype("Int64")
_csv = _csv.dropna(subset=["Time"])

# Mean across runs for every (Size, Method, Procs, Nodes, Halo) cell.
_means = _csv.groupby(["Size", "Method", "Procs", "Nodes", "Halo"])["Time"].mean()

sizes = [128, 512, 1024, 2048, 4096]
procs = [1, 2, 4, 16, 32]

### Average execution times

In [19]:
def _t(method, size, p, nodes=1, halo=1):
    key = (size, method, p, nodes, halo)
    return _means[key] if key in _means.index else np.nan

# Average execution time per (Size, Procs) on a single node, halo=1.
# First column "seq" is the pure sequential baseline.
# the remaining columns are the MPI variant run with P = 1, 2, 4, 16, 32 ranks.
def _time_table(method):
    cols = ["seq"] + procs
    df = pd.DataFrame(index=sizes, columns=cols, dtype=float)
    df.index.name = "Size"
    for s in sizes:
        df.loc[s, "seq"] = _t("seq", s, 1)
        for p in procs:
            df.loc[s, p] = _t(method, s, p)
    df.attrs["name"] = f"Average execution time (s) for method: {method}"
    return df
display_list([_time_table("row"), _time_table("block")], axis=1)

,seq,1,2,4,16,32
Size,,,,,,
128,1.18,1.14,0.6,0.36,0.14,
512,19.49,19.31,9.22,5.56,1.24,0.71
1024,76.95,75.05,37.43,22.24,7.24,2.93
2048,312.09,294.96,147.92,73.45,19.31,10.29
4096,1557.65,1587.7,790.32,392.97,96.22,48.49
,seq,1,2,4,16,32
Size,,,,,,
128,1.18,1.01,0.51,0.26,0.07,0.04
512,19.49,16.13,8.07,4.15,1.03,0.53


### Block vs row speed-up

Same grid size and process count; values are $T_{\mathrm{row}} / T_{\mathrm{block}}$ (values $> 1$ mean block is faster).

In [20]:
# Speed-up where t_s is the sequential baseline.
def _speedup_table(method):
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        ts = _t("seq", s, 1)
        for p in procs:
            tp = _t(method, s, p)
            df.loc[s, p] = (ts / tp) if (tp and tp > 0) else np.nan
    df.attrs["name"] = f"Speed-up for method: {method}"
    return df

# Speed-up of block over row at the same (Size, Procs): T_row / T_block.
def _block_vs_row_speedup_table():
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        for p in procs:
            t_row = _t("row", s, p)
            t_block = _t("block", s, p)
            df.loc[s, p] = (t_row / t_block) if (t_row and t_block and t_block > 0) else np.nan
    df.attrs["name"] = "Speed-up: block vs row"
    return df

display_list([_speedup_table("row"), _speedup_table("block")], axis=1, minmax="max")
display_list([_block_vs_row_speedup_table()], axis=1, minmax="max")

Procs,1,2,4,16,32
Size,,,,,
128,1.03,1.98,3.26,8.68,
512,1.01,2.11,3.51,15.69,27.37
1024,1.03,2.06,3.46,10.62,26.25
2048,1.06,2.11,4.25,16.16,30.33
4096,0.98,1.97,3.96,16.19,32.12
Procs,1,2,4,16,32
Size,,,,,
128,1.16,2.32,4.49,16.94,30.24
512,1.21,2.41,4.7,18.89,36.99


Procs,1,2,4,16,32
Size,,,,,
128,1.13,1.17,1.38,1.95,
512,1.2,1.14,1.34,1.2,1.35
1024,1.16,1.16,1.36,1.76,1.43
2048,1.14,1.15,1.14,1.18,1.27
4096,1.54,1.53,1.52,1.44,1.46


### Single-node vs two-node runs

Two-node benchmarks exist only for **N = 1024, 4096** with **P = 16, 32** (same total rank count). Speed-up below is $T_{\mathrm{1\,node}} / T_{\mathrm{2\,nodes}}$; values $> 1$ mean the 2-node run was faster.

In [26]:
# 1-node vs 2-nodes for the same total process count.
def _nodes_table(method, nodes):
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        for p in procs:
            df.loc[s, p] = _t(method, s, p, nodes=nodes)
    suffix = f"{nodes} node" + ("s" if nodes > 1 else "")
    df.attrs["name"] = f"Average execution time (s), method = {method}, {suffix}"
    return df

# Speed-up of 2-node runs over 1-node at the same (Size, Procs): T_1node / T_2node.
def _nodes_speedup_table(method):
    df = pd.DataFrame(index=sizes, columns=procs, dtype=float)
    df.index.name = "Size"
    df.columns.name = "Procs"
    for s in sizes:
        for p in procs:
            t1 = _t(method, s, p, nodes=1)
            t2 = _t(method, s, p, nodes=2)
            df.loc[s, p] = (t1 / t2) if (t1 and t2 and t2 > 0) else np.nan
    df.attrs["name"] = f"Speed-up: 2 nodes vs 1 node, method: {method}"
    return df

print("Method = row")
display_list([_nodes_table("row", 1), _nodes_table("row", 2)])
display_list([_nodes_speedup_table("row")], axis=1, minmax="max")

print("Method = block")
display_list([_nodes_table("block", 1), _nodes_table("block", 2)])
display_list([_nodes_speedup_table("block")], axis=1, minmax="max")

Method = row


Procs,1,2,4,16,32
Size,,,,,
128,1.14,0.6,0.36,0.14,
512,19.31,9.22,5.56,1.24,0.71
1024,75.05,37.43,22.24,7.24,2.93
2048,294.96,147.92,73.45,19.31,10.29
4096,1587.7,790.32,392.97,96.22,48.49
Procs,1,2,4,16,32
Size,,,,,
128,,,,,
512,,,,,


Procs,1,2,4,16,32
Size,,,,,
128,,,,,
512,,,,,
1024,,,,0.89,1.0
2048,,,,,
4096,,,,0.99,1.03


Method = block


Procs,1,2,4,16,32
Size,,,,,
128,1.01,0.51,0.26,0.07,0.04
512,16.13,8.07,4.15,1.03,0.53
1024,64.49,32.26,16.4,4.11,2.04
2048,258.11,129.1,64.59,16.38,8.11
4096,1032.91,516.59,258.57,66.63,33.29
Procs,1,2,4,16,32
Size,,,,,
128,,,,,
512,,,,,


Procs,1,2,4,16,32
Size,,,,,
128,,,,,
512,,,,,
1024,,,,1.01,0.98
2048,,,,,
4096,,,,1.03,1.02


### Wide-halo communication-overhead 

In [22]:
# Wide-halo K sweep (bonus). Halo width = K * R, exchange every K iterations.
# Split by node count so 1-node and 2-node K curves can be compared side by side.
_wide = _csv[_csv["Method"] == "row_wide"].copy()
tables = []
for n in sorted(_wide["Nodes"].dropna().unique()):
    sub = _wide[_wide["Nodes"] == n]
    df = sub.groupby(["Size", "Procs", "Halo"])["Time"].mean().unstack("Halo")
    df.columns.name = "K"
    suffix = f"{int(n)} node" + ("s" if int(n) > 1 else "")
    df.attrs["name"] = f"Mean time (s) per K for {suffix}"
    tables.append(df)
display_list(tables, axis=1)

,K,1,2,4,8
Size,Procs,,,,
2048,16,19.49,21.12,24.96,32.68
4096,32,49.98,54.86,68.48,108.59
,K,1,2,4,8
Size,Procs,,,,
2048,16,21.21,22.56,25.3,33.04
4096,32,50.73,53.94,68.5,90.17
